# 01 · Veri Hazırlama — RVL-CDIP (4 sınıf)

**Bitirme Projesi:** OCR tabanlı belge sınıflandırma ve içerik çıkarımı
**Sınıflar:** E-posta · Form · Fatura · CV (resume)

Bu notebook:
1. RVL-CDIP'in dengeli alt kümesini (`jinhybr/rvl_cdip_400_train_val_test`, ~1.1 GB) indirir
2. Sadece 4 hedef sınıfı filtreler → **sınıf başı 400 train / 100 val / 100 test = 2.400 görüntü**
3. Görüntüleri Google Drive'a PNG olarak kaydeder ve `metadata.csv` oluşturur

> Tam RVL-CDIP 38 GB olduğu için Colab'da indirmek pratik değil. Bu alt küme orijinal train/val/test ayrımını korur.
> Daha fazla veri gerekirse bölüm 6'ya bakın.

## 1. Kurulum

In [ ]:
!pip -q install -U datasets

## 2. Drive ve yollar

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
ROOT = Path("/content/drive/MyDrive/bitirme_ocr")   # tüm proje çıktıları burada
DATA = ROOT / "data"
IMG_DIR = DATA / "images"
OCR_DIR = DATA / "ocr"
for p in (IMG_DIR, OCR_DIR):
    p.mkdir(parents=True, exist_ok=True)

# Projedeki 4 sınıf (sıra = model etiket id'leri 0..3)
CLASSES = ["email", "form", "invoice", "resume"]
print("Proje klasörü:", ROOT)

## 3. Veri setini indir

In [ ]:
from datasets import load_dataset

ds = load_dataset("jinhybr/rvl_cdip_400_train_val_test")
label_names = ds["train"].features["label"].names
print(ds)
print("RVL-CDIP sınıfları:", label_names)

## 4. Sadece 4 sınıfı filtrele

In [ ]:
# RVL-CDIP adı -> bizim sınıf adımız
RVL_TO_OURS = {"email": "email", "form": "form", "invoice": "invoice", "resume": "resume"}
keep_ids = {label_names.index(rvl): ours for rvl, ours in RVL_TO_OURS.items()}
print("Tutulacak RVL id'leri:", keep_ids)

# input_columns sayesinde görüntüler decode edilmeden hızlı filtrelenir
sub = ds.filter(lambda lbl: lbl in keep_ids, input_columns="label")
for split in sub:
    print(split, len(sub[split]))

## 5. Görüntüleri Drive'a kaydet + metadata.csv

In [ ]:
import pandas as pd
from tqdm.auto import tqdm

rows = []
for split in sub:
    for i, ex in enumerate(tqdm(sub[split], desc=split)):
        cls = keep_ids[ex["label"]]
        out_dir = IMG_DIR / split / cls
        out_dir.mkdir(parents=True, exist_ok=True)
        path = out_dir / f"{split}_{cls}_{i:05d}.png"
        img = ex["image"].convert("L")          # RVL-CDIP zaten gri tonlamalı
        if not path.exists():                   # tekrar çalıştırmada atla
            img.save(path)
        rows.append({
            "path": str(path.relative_to(DATA)),
            "split": split,
            "label": cls,
            "label_id": CLASSES.index(cls),
            "width": img.width,
            "height": img.height,
        })

meta = pd.DataFrame(rows)
meta.to_csv(DATA / "metadata.csv", index=False)
print("Kaydedildi:", DATA / "metadata.csv", meta.shape)

## 6. Kontrol: sınıf dağılımı

In [ ]:
meta = pd.read_csv(DATA / "metadata.csv")
pd.crosstab(meta["label"], meta["split"], margins=True)

## 7. Örnek görüntüler

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image

fig, axes = plt.subplots(len(CLASSES), 4, figsize=(14, 18))
for r, cls in enumerate(CLASSES):
    samples = meta[(meta.label == cls) & (meta.split == "train")].sample(4, random_state=42)
    for c, (_, row) in enumerate(samples.iterrows()):
        axes[r, c].imshow(Image.open(DATA / row["path"]), cmap="gray")
        axes[r, c].set_title(cls, fontsize=12)
        axes[r, c].axis("off")
plt.tight_layout()
plt.savefig(ROOT / "ornek_belgeler.png", dpi=120)   # rapor için
plt.show()

## 8. Görüntü boyut istatistikleri

In [ ]:
meta.groupby("label")[["width", "height"]].describe().round(0)

## Not: Daha fazla veri gerekirse

| Seçenek | Boyut | Not |
|---|---|---|
| `jinhybr/rvl_cdip_400_train_val_test` (bu notebook) | 600 / sınıf | Hızlı, Colab'a uygun |
| `chainyo/rvl-cdip-invoice` vb. sınıf bazlı alt kümeler | ~25.000 / sınıf | Sadece bazı sınıflar mevcut |
| Orijinal RVL-CDIP | 25.000 / sınıf, 38 GB | Colab disk sınırına yakın |

Sonraki adım → **02_ocr_pipeline.ipynb**